In [ ]:
from loader import mol_to_graph_data_obj_simple
from model import GNN_graphpred
import torch
from rdkit import Chem

from torch_geometric.data import Data

def ensure_batch(data: Data) -> Data:
    if getattr(data, 'batch', None) is None:
        data.batch = torch.zeros(data.x.size(0), dtype=torch.long)
    return data

In [ ]:
checkpoint_path = "model_gin/supervised_contextpred.pth" 

device = "cpu"


model = GNN_graphpred(
    num_layer=5,
    emb_dim=300,
    num_tasks=1,
)

model.from_pretrained(checkpoint_path)

model.to(device)
model.eval()



GNN_graphpred(
  (gnn): GNN(
    (x_embedding1): Embedding(120, 300)
    (x_embedding2): Embedding(3, 300)
    (gnns): ModuleList(
      (0-4): 5 x GINConv()
    )
    (batch_norms): ModuleList(
      (0-4): 5 x BatchNorm1d(300, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (graph_pred_linear): Linear(in_features=300, out_features=1, bias=True)
)

In [ ]:
@torch.no_grad()
def get_molecule_embeddings(mol: Chem.Mol) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Takes as input a SMILES string and returns a vector embedding for 
    the whole graph and each node's embeddings, one vector per node, 
    as a node embeddings matrix.
    """
    data = mol_to_graph_data_obj_simple(mol)
    data = ensure_batch(data).to(device)
    out = model(data)
    graph_emb = out["graph"]
    node_emb  = out["node"]
    return graph_emb, node_emb

In [4]:
smiles_list = [
    "CCO",         # ethanol
    "c1ccccc1O",   # phenol
    "CC(=O)O",     # acetic acid
]

embeddings_per_mol = {}
for smi in smiles_list:
    embeddings_per_mol[smi] = get_molecule_embeddings(smi)

SMILES: CCO
Graph embedding shape: torch.Size([1, 300])
Node embedding shape : torch.Size([3, 300])
SMILES: c1ccccc1O
Graph embedding shape: torch.Size([1, 300])
Node embedding shape : torch.Size([7, 300])
SMILES: CC(=O)O
Graph embedding shape: torch.Size([1, 300])
Node embedding shape : torch.Size([4, 300])


In [9]:
first = smiles_list[0]
graph_vec = embeddings_per_mol[first]['graph']  # tensor of shape [1, output_dim]
node_matrix = embeddings_per_mol[first]['node'] # tensor [num_atoms, emb_dim]

node_matrix

tensor([[ 1.4086e-02, -5.5777e-02, -5.5727e-02,  1.4093e-01,  9.2146e-02,
         -1.9372e-02,  3.0496e-02,  8.4878e-02, -4.4426e-02, -2.2303e-03,
         -4.8644e-02, -5.0792e-03, -9.2918e-02,  1.1366e-01, -8.9929e-02,
         -3.5270e-02,  1.7687e-02,  1.4464e-01, -7.4153e-02,  8.3692e-02,
          1.4368e-01,  1.0515e-01,  6.1848e-02,  1.5702e-01,  1.1516e-01,
         -5.3472e-02, -4.5693e-02, -2.6352e-02, -1.9974e-02, -6.5208e-02,
         -1.6604e-01,  1.3329e-01,  3.1410e-02, -2.2186e-02, -8.5646e-03,
          1.3848e-02,  1.1463e-01,  7.2754e-02, -2.0827e-02, -6.6610e-02,
          5.4670e-02, -2.2247e-01,  8.1789e-02, -8.1186e-02,  3.3692e-02,
         -1.4345e-01, -3.7394e-03, -3.1083e-02,  3.3237e-02,  1.2029e-02,
         -3.3465e-02, -6.4525e-03,  1.0551e-01, -7.7029e-03,  1.9138e-01,
          1.1541e-01, -4.5414e-02,  3.2594e-02,  4.9468e-02, -3.8680e-01,
         -1.5574e-01, -5.8320e-02, -7.9951e-02,  9.8181e-03, -3.5043e-02,
         -2.5932e-02, -1.4524e-02, -3.